# Download & Convert All Google Sheets Revisions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/google-sheets-history-cQ6AV/notebooks/download-all-revisions.ipynb)

Downloads every revision of the UBL 2.5 Google Sheets as ODS,
converts new unique content states to `.gc`, gzips, and saves to Drive.

**Crash-resilient:** re-run and it picks up where it left off.

**No repo clone** — fetches only the 4 tool files (~5 MB) from GitHub.

## For each revision (newest → oldest)

1. Download as ODS via Drive API v2
2. Hash `content.xml` to detect unique spreadsheet states
3. If **new** unique state → convert ODS → `.gc` via Saxon/Crane → gzip
4. Save `rev-{id}.ods.gz` (and `.gc.gz` when converted) to Drive
5. Record everything in `manifest-{sheet}.json`

## Output

```
Drive: ubl-gc-revisions/
├── ubl25_library/
│   ├── rev-2005.ods.gz           (every revision)
│   ├── rev-2004.ods.gz
│   └── ...
├── ubl25_library-gc/
│   ├── rev-2005.entities.gc.gz   (only new unique states)
│   ├── rev-2005.endorsed.gc.gz
│   └── ...
├── ubl25_documents/
│   └── ...
├── ubl25_documents-gc/
│   └── ...
├── manifest-ubl25_library.json
└── manifest-ubl25_documents.json
```

In [ ]:
# === Step 0: Auth ===
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

In [ ]:
# === Step 1: Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

In [ ]:
# === Step 2: Fetch tool files from GitHub (no repo clone!) ===
import subprocess, os, shutil, tempfile

TOOLS_DIR = Path('/content/tools')
TOOLS_DIR.mkdir(exist_ok=True)

BRANCH = 'claude/google-sheets-history-cQ6AV'
RAW = f'https://raw.githubusercontent.com/kduvekot/ubl-gc/{BRANCH}'

TOOL_URLS = {
    'saxon9he.jar':         f'{RAW}/history/tools/saxon9he/saxon9he.jar',
    'Crane-ods2obdgc.xsl':  f'{RAW}/history/tools/Crane-ods2obdgc/Crane-ods2obdgc.xsl',
    'massageModelName.xml': f'{RAW}/work-sheets/scripts/massageModelName.xml',
    'gc2endorsed.xsl':      f'{RAW}/work-sheets/scripts/gc2endorsed.xsl',
}

for name, url in TOOL_URLS.items():
    dest = TOOLS_DIR / name
    if dest.exists() and dest.stat().st_size > 100:
        print(f'  [skip] {name} ({dest.stat().st_size:,} bytes)')
        continue
    print(f'  Downloading {name}...', end=' ')
    r = subprocess.run(['wget', '-q', '-O', str(dest), url],
                       capture_output=True, timeout=60)
    if r.returncode != 0 or not dest.exists():
        raise RuntimeError(f'Failed to download {name}')
    print(f'{dest.stat().st_size:,} bytes')

SAXON_JAR     = str(TOOLS_DIR / 'saxon9he.jar')
CRANE_XSL     = str(TOOLS_DIR / 'Crane-ods2obdgc.xsl')
MASSAGE_XML   = str(TOOLS_DIR / 'massageModelName.xml')
GC2ENDORSED   = str(TOOLS_DIR / 'gc2endorsed.xsl')

for f in [SAXON_JAR, CRANE_XSL, MASSAGE_XML, GC2ENDORSED]:
    assert os.path.getsize(f) > 100, f'Bad download: {f}'

# Verify Java is available (Colab has it by default)
!java -version 2>&1 | head -1
print('\nAll tools ready!')

In [ ]:
# === Step 3: Configuration & helpers ===
import json, hashlib, gzip, time, zipfile, io
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from collections import Counter

SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

ODS_MIME     = 'application/x-vnd.oasis.opendocument.spreadsheet'
ODS_MIME_ALT = 'application/vnd.oasis.opendocument.spreadsheet'
SHEET_REGEX  = r'^([Ll]($|[^o].*|o($|[^g].*|g($|[^s].*))))|^[^Ll].*'


def api_get(url, binary=False):
    """Authenticated GET with retry + exponential backoff."""
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                return resp.status, data if binary else json.loads(data)
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  Retry ({e.code}), waiting {wait}s...')
                time.sleep(wait)
                continue
            return e.code, e.read().decode(errors='replace')
        except Exception as e:
            if attempt < 3:
                wait = 2 ** (attempt + 1)
                print(f'  Error: {e}, retrying in {wait}s...')
                time.sleep(wait)
                continue
            return 0, str(e)
    return 0, 'max retries exceeded'


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS (ZIP) and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


def download_revision_ods(file_id, rev_id):
    """Download a single revision as ODS. Returns bytes or None."""
    v2_url = (f'https://www.googleapis.com/drive/v2/files/{file_id}'
              f'/revisions/{rev_id}')
    status, data = api_get(v2_url)
    if status != 200 or not isinstance(data, dict):
        return None

    export_links = data.get('exportLinks', {})
    ods_url = export_links.get(ODS_MIME) or export_links.get(ODS_MIME_ALT)
    if not ods_url:
        return None

    time.sleep(0.3)
    status, ods_data = api_get(ods_url, binary=True)
    if status != 200 or not isinstance(ods_data, bytes):
        return None
    return ods_data


# --- Conversion helpers ---

def make_ident_xml(tmpdir, endorsed=False):
    """Write identification XML (fixed CSD02 stage)."""
    sfx  = '-Endorsed' if endorsed else ''
    nsfx = ' Endorsed' if endorsed else ''
    usfx = ':ENDORSED' if endorsed else ''
    fsfx = '-Endorsed' if endorsed else ''
    xml = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<Identification>\n'
        f'  <ShortName>UBL-2.5-CSD02{sfx}</ShortName>\n'
        f'  <LongName>UBL 2.5 CSD02{nsfx} Business Entity Summary</LongName>\n'
        '  <Version>2.5</Version>\n'
        f'  <CanonicalUri>urn:oasis:names:specification:ubl:BIE{usfx}</CanonicalUri>\n'
        f'  <CanonicalVersionUri>urn:oasis:names:specification:ubl:BIE{usfx}:2.5</CanonicalVersionUri>\n'
        f'  <LocationUri>http://docs.oasis-open.org/ubl/csd02-UBL-2.5/mod/UBL-Entities-2.5{fsfx}.gc</LocationUri>\n'
        '  <Agency>\n'
        '     <LongName xml:lang="en">OASIS Universal Business Language</LongName>\n'
        '     <Identifier>UBL</Identifier>\n'
        '  </Agency>\n'
        '</Identification>'
    )
    fname = 'ident-UBL-Endorsed.xml' if endorsed else 'ident-UBL.xml'
    path = os.path.join(tmpdir, fname)
    with open(path, 'w') as f:
        f.write(xml)
    return path


def run_saxon(args):
    """Run Saxon, return True on success."""
    cmd = ['java', '-jar', SAXON_JAR] + args
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    return result.returncode == 0


def convert_to_gc(lib_ods_bytes, doc_ods_bytes):
    """Convert library + documents ODS → .gc files.
    Returns (entities_bytes, endorsed_bytes) or (None, None)."""
    tmpdir = tempfile.mkdtemp()
    try:
        lib_path = os.path.join(tmpdir, 'UBL-Library-Google.ods')
        doc_path = os.path.join(tmpdir, 'UBL-Documents-Google.ods')
        Path(lib_path).write_bytes(lib_ods_bytes)
        Path(doc_path).write_bytes(doc_ods_bytes)
        shutil.copy2(MASSAGE_XML, os.path.join(tmpdir, 'massageModelName.xml'))

        ods_list = f'{lib_path},{doc_path}'
        make_ident_xml(tmpdir, endorsed=False)
        make_ident_xml(tmpdir, endorsed=True)

        entities_out = os.path.join(tmpdir, 'entities.gc')
        endorsed_out = os.path.join(tmpdir, 'endorsed.gc')
        raw_endorsed = os.path.join(tmpdir, 'raw-endorsed.gc')

        # 1/3: Entities
        if not run_saxon([
            f'-xsl:{CRANE_XSL}', f'-o:{entities_out}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident-UBL.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ]):
            return None, None

        # 2/3: Raw endorsed
        if not run_saxon([
            f'-xsl:{CRANE_XSL}', f'-o:{raw_endorsed}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident-UBL-Endorsed.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ]):
            return None, None

        # 3/3: Filter endorsed
        if not run_saxon([
            f'-o:{endorsed_out}', f'-s:{raw_endorsed}',
            f'-xsl:{GC2ENDORSED}',
        ]):
            return None, None

        return Path(entities_out).read_bytes(), Path(endorsed_out).read_bytes()
    finally:
        shutil.rmtree(tmpdir)


print('All helpers ready')

In [ ]:
# === Step 4: List all revisions of both sheets ===
all_revisions = {}

for sheet_key, file_id in SHEETS.items():
    print(f'\n=== {sheet_key} ===')
    revisions = []
    page_token = None

    while True:
        url = (f'https://www.googleapis.com/drive/v2/files/{file_id}'
               f'/revisions?maxResults=1000')
        if page_token:
            url += f'&pageToken={page_token}'

        status, data = api_get(url)
        if status != 200:
            print(f'  ERROR: HTTP {status}')
            break

        items = data.get('items', [])
        revisions.extend(items)

        page_token = data.get('nextPageToken')
        if not page_token:
            break
        print(f'  ... {len(revisions)} so far')

    all_revisions[sheet_key] = revisions
    print(f'  Total: {len(revisions)} revisions')
    if revisions:
        print(f'  First: rev-{revisions[0]["id"]} '
              f'({revisions[0].get("modifiedDate", "?")})')
        print(f'  Last:  rev-{revisions[-1]["id"]} '
              f'({revisions[-1].get("modifiedDate", "?")})')

print(f'\nGrand total: {sum(len(v) for v in all_revisions.values())} revisions')

## Step 5: Download + Convert Loop

**Set `SHEET_KEY`** to choose which sheet to process.

Works backwards (newest → oldest). For each revision:
- Downloads ODS, gzips, saves `rev-{id}.ods.gz` to Drive
- Hashes `content.xml` to detect unique spreadsheet states
- On **first occurrence** of a new state: converts to `.gc` via
  Saxon/Crane, gzips, saves `rev-{id}.entities.gc.gz` + `.endorsed.gc.gz`
- On **repeat** state: skips conversion (same content = same `.gc`)

The conversion pairs the target sheet's revision with the **latest**
revision of the other sheet. This is enough to find unique `.gc`
states and match against known CI run outputs.

**Resume-safe:** skips revisions already on Drive.

In [ ]:
# ============================================================
# CHOOSE WHICH SHEET TO PROCESS
# Run once with 'ubl25_library', then with 'ubl25_documents'
# ============================================================
SHEET_KEY = 'ubl25_library'
# SHEET_KEY = 'ubl25_documents'
# ============================================================

file_id = SHEETS[SHEET_KEY]
revisions = all_revisions[SHEET_KEY]

# --- Directories ---
ods_dir = DRIVE_DIR / SHEET_KEY
gc_dir  = DRIVE_DIR / f'{SHEET_KEY}-gc'
ods_dir.mkdir(exist_ok=True)
gc_dir.mkdir(exist_ok=True)

# --- Resume from manifest ---
manifest_path = DRIVE_DIR / f'manifest-{SHEET_KEY}.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    done_ids = {str(r['id']) for r in manifest.get('revisions', [])}
    print(f'Resuming: {len(done_ids)} already in manifest')
else:
    manifest = {'sheet_id': file_id, 'sheet_key': SHEET_KEY, 'revisions': []}
    done_ids = set()

# Existing files as fallback
existing_ods = set(f.name for f in ods_dir.glob('rev-*.ods.gz'))
existing_gc  = set(f.name for f in gc_dir.glob('rev-*.entities.gc.gz'))
print(f'ODS on Drive: {len(existing_ods)}, GC on Drive: {len(existing_gc)}')

# Track unique content hashes seen so far
seen_hashes = set()
hash_to_rev = {}  # content_hash → first rev_id that has .gc
for r in manifest.get('revisions', []):
    h = r.get('content_hash')
    if h:
        seen_hashes.add(h)
        if r.get('gc_entities_hash') and h not in hash_to_rev:
            hash_to_rev[h] = str(r['id'])

# --- Download partner sheet's latest revision ---
partner_key = ('ubl25_documents' if SHEET_KEY == 'ubl25_library'
               else 'ubl25_library')
partner_id  = SHEETS[partner_key]
partner_revs = all_revisions[partner_key]
latest_partner = partner_revs[-1]

print(f'\nDownloading partner ({partner_key}) latest: '
      f'rev-{latest_partner["id"]}...', end=' ')
partner_ods = download_revision_ods(partner_id, latest_partner['id'])
assert partner_ods, 'Failed to download partner ODS'
print(f'{len(partner_ods):,} bytes')

# --- Main loop ---
rev_list = list(reversed(revisions))  # newest first
total = len(rev_list)
skipped = 0
downloaded = 0
converted = 0
errors = 0

print(f'\nProcessing {total} revisions of {SHEET_KEY} (newest → oldest)\n')

for i, rev in enumerate(rev_list):
    rev_id = str(rev['id'])
    modified = rev.get('modifiedDate', '?')

    # --- Skip if done ---
    if rev_id in done_ids:
        skipped += 1
        continue

    gz_path = ods_dir / f'rev-{rev_id}.ods.gz'
    if gz_path.exists() and gz_path.stat().st_size > 0:
        skipped += 1
        done_ids.add(rev_id)
        continue

    pct = (i + 1) / total * 100
    print(f'[{i+1}/{total} {pct:.0f}%] rev-{rev_id} ({modified})', end=' ')

    # --- Download ODS ---
    ods_data = download_revision_ods(file_id, rev_id)
    if not ods_data:
        print('ERROR: download failed')
        errors += 1
        time.sleep(1)
        continue

    # --- Hash content ---
    content_hash = ods_content_hash(ods_data)
    is_new = content_hash and content_hash not in seen_hashes
    if content_hash:
        seen_hashes.add(content_hash)

    # --- Save ODS.gz to Drive ---
    gz_data = gzip.compress(ods_data, compresslevel=6)
    gz_path.write_bytes(gz_data)

    # --- Build manifest entry ---
    entry = {
        'id': rev_id,
        'modifiedDate': modified,
        'ods_size': len(ods_data),
        'gz_size': len(gz_data),
        'content_hash': content_hash,
    }

    print(f'{len(ods_data):,}b', end='')

    # --- Convert if new unique state ---
    if is_new:
        if SHEET_KEY == 'ubl25_library':
            ent_bytes, end_bytes = convert_to_gc(ods_data, partner_ods)
        else:
            ent_bytes, end_bytes = convert_to_gc(partner_ods, ods_data)

        if ent_bytes and end_bytes:
            ent_hash = hashlib.sha256(ent_bytes).hexdigest()
            end_hash = hashlib.sha256(end_bytes).hexdigest()

            ent_gz = gzip.compress(ent_bytes, compresslevel=6)
            end_gz = gzip.compress(end_bytes, compresslevel=6)
            (gc_dir / f'rev-{rev_id}.entities.gc.gz').write_bytes(ent_gz)
            (gc_dir / f'rev-{rev_id}.endorsed.gc.gz').write_bytes(end_gz)

            entry['gc_entities_hash'] = ent_hash
            entry['gc_endorsed_hash'] = end_hash
            hash_to_rev[content_hash] = rev_id
            converted += 1

            print(f' NEW .gc ent={ent_hash[:12]}... end={end_hash[:12]}...')
        else:
            print(' NEW (conversion FAILED)')
    else:
        # Link to the first revision that shares this content
        ref = hash_to_rev.get(content_hash)
        if ref:
            entry['gc_same_as'] = ref
        print(f' (same as rev-{ref})' if ref else '')

    manifest['revisions'].append(entry)
    done_ids.add(rev_id)
    downloaded += 1

    # --- Save manifest periodically ---
    if downloaded % 25 == 0:
        manifest_path.write_text(json.dumps(manifest, indent=2))
        print(f'  --- saved: {downloaded} new, {skipped} skip, '
              f'{converted} converted, {len(seen_hashes)} unique ---')

    time.sleep(0.5)

# --- Final save ---
manifest['unique_states'] = len(seen_hashes)
manifest['total_revisions'] = total
manifest['converted'] = converted
manifest['partner_sheet'] = partner_key
manifest['partner_rev'] = str(latest_partner['id'])
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f'\n{"="*60}')
print(f'DONE: {SHEET_KEY}')
print(f'  Downloaded:     {downloaded}')
print(f'  Skipped:        {skipped}')
print(f'  Errors:         {errors}')
print(f'  Unique states:  {len(seen_hashes)}')
print(f'  Converted:      {converted}')
print(f'  Manifest:       {manifest_path}')

## Step 6: Analyze Results

In [ ]:
for sheet_key in SHEETS:
    mp = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if not mp.exists():
        print(f'{sheet_key}: not yet downloaded')
        continue

    m = json.loads(mp.read_text())
    revs = m.get('revisions', [])
    hash_counts = Counter(
        r['content_hash'] for r in revs if r.get('content_hash')
    )

    print(f'\n{"="*60}')
    print(f'{sheet_key}: {len(revs)} downloaded, '
          f'{len(hash_counts)} unique states, '
          f'{m.get("converted", "?")} converted')
    print(f'{"="*60}')

    # Unique states with revision ranges
    print(f'\nUnique states (most common first):')
    for rank, (h, count) in enumerate(hash_counts.most_common(), 1):
        matching = sorted(
            [r for r in revs if r.get('content_hash') == h],
            key=lambda r: int(r['id'])
        )
        first = matching[0]
        last = matching[-1]
        gc_tag = ''
        if first.get('gc_entities_hash'):
            gc_tag = f' .gc={first["gc_entities_hash"][:12]}...'
        elif first.get('gc_same_as'):
            gc_tag = f' .gc→rev-{first["gc_same_as"]}'
        print(f'  {rank:3d}. {h[:16]}... x{count:4d}  '
              f'rev-{first["id"]} to rev-{last["id"]}{gc_tag}')

    # Cross-reference known CI run revisions
    known = {
        'ubl25_library': {
            '1843': 'V1/V2', '1868': 'V3/V4',
            '1999': 'V5/V6', '2005': 'V7-V10',
        },
        'ubl25_documents': {
            '1793': 'V1/V2', '1803': 'V3', '1983': 'V4',
            '2190': 'V5-V7', '2200': 'V8', '2204': 'V9/V10',
        },
    }.get(sheet_key, {})

    if known:
        print(f'\nKnown CI-run revisions:')
        for rev_id, label in known.items():
            entry = next(
                (r for r in revs if str(r['id']) == rev_id), None
            )
            if entry:
                h = entry.get('content_hash', '?')
                count = hash_counts.get(h, 0)
                gc_info = ''
                if entry.get('gc_entities_hash'):
                    gc_info = f', .gc={entry["gc_entities_hash"][:12]}...'
                print(f'  rev-{rev_id} ({label}): '
                      f'{h[:16]}... ({count} share this){gc_info}')
            else:
                print(f'  rev-{rev_id} ({label}): not downloaded yet')

## What Next

After both sheets are done:

1. **Match CI runs** — compare `.gc` hashes against known CI workflow
   outputs from V1-V10 to find exact revision IDs
2. **Build timeline** — pair library + documents revisions by timestamp
   to find actual (lib, doc) combinations used in each CI run
3. **Convert proper pairs** — once pairing is known, convert with
   correct partner revisions for exact CI-match `.gc` output